# D1.7 · Drift monitoring

**Function D — The Agentic SOC → The Agentic SOC — Detection**  ·  *Security of AI*

Builds on **[D1.6 · Distinguishing agent from human](https://spbreed.github.io/cyber-commons/lessons/D1.6.html)**.

| | |
|---|---|
| Tools used | promptfoo, GLM-4.6, Claude Haiku 4.5 |

## What this lesson is

**What it covers.** Change the model underneath and catch the detection regression.

**Why a security engineer needs it.** A detection that worked last month is silently degraded. The control it builds is: watch model updates, prompt changes, index refreshes, tool versions.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Nothing was attacked. The model was upgraded, a prompt was edited, a tool changed its output format — and the behaviour of the system moved. Drift is the failure mode with no adversary, and it is far more common than the ones with one.

> **At CyberTravels.** Nothing was attacked. The model provider upgraded, Alex edited a prompt, the tool manifest changed — and CyberTravels' baseline moved underneath every detection built on it.

## 2 · The framework

```
   nothing was attacked

   model upgraded ----+
   prompt edited  ----+---> behaviour moves ---> baselines stale
   tool changed   ----+                          detections silent

   drift is the failure mode with no adversary, and the common one
   the control: a fixed probe suite, run on every change
```

Drift monitoring exists because an agent's behaviour changes **without a code
change**. A new model version, an edited prompt, an added tool — none of these
pass through the change management process built for code, and all of them
invalidate the testing your controls were signed off against.

That is the precise claim: the control was tested against a behaviour that no
longer exists. It has not failed; it is *unevidenced*, which is a different and
more honest state.

Two things are needed:

1. A **signed-off baseline** — what normal looked like when the control passed.
2. A **freshness window** on the control test, derived from how fast the thing
   it tests actually drifts.

E1.7 turns the second into a compliance posture. This lesson produces the signal.

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">change surface</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">in change management?</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">what happens today</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">application code</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">yes</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">pull request, review, CI</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">agent prompt</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>no</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">edited in a console</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">tool manifest</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>no</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">a config change, with no threat-model diff</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">model version</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>no</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">provider-side; you may not be told</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">policy</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600">yes</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">if it is in git — often it is not</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">approval settings</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600"><b>no</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">a toggle in an admin UI</td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">Four of six surfaces bypass change management entirely. Drift is the failure mode with no adversary, and this table is why it is also the failure mode with no ticket.</div>

## 3 · The procedure, as a skill

Four of six things that change an agent's behaviour never reach change management. The skill counts them, then tracks drift across a quarter and attributes the rise that coincides with the model upgrade — and the one that does not.

### The skill — [`skills/detection/behavioural-drift-monitor/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/behavioural-drift-monitor/SKILL.md)

```yaml
name: behavioural-drift-monitor
description: >-
  Track an agent's drift from its signed-off baseline across a quarter and count
  how many of the surfaces that change its behaviour bypass change management.
  Use when an agent behaves differently than it was approved to, with no code
  change to point at.
allowed-tools: Read, Grep, Glob
```

# Four of six things that change an agent are not code changes

The model version, the prompt, the tool manifest and the approval settings all
change what an agent does, and none of them generates a change record. So drift
is the only signal that the approved system and the running system have diverged,
and it has to be measured rather than inferred from a changelog.

## When to use this

Continuously, for any agent under governance — and specifically after a provider
announces a model update.

## Procedure

**1 — Enumerate the change surfaces.** Code, model version, system prompt, tool
manifest, approval settings, retrieval corpus. For each, record whether a change
produces a record anybody reviews.

**2 — Count the ones that bypass change management.** Four of six is typical.
That count is the argument for measuring drift at all.

**3 — Fix a baseline at sign-off.** Tools, resources, rate, refusal rate — and
the model version and prompt hash alongside, so a later drift can be attributed.

**4 — Compute drift per period and attribute each rise.** A jump that coincides
with a model upgrade is a different conversation from a jump with no
corresponding event, and the second is the one to escalate.

**5 — Set a tolerance and a consequence.** Beyond tolerance: re-attest, or
revert. A drift figure with no consequence is a chart.

## Example

**Input** — the fixture committed at the top of [`scripts/behavioural_drift_monitor.py`](scripts/behavioural_drift_monitor.py). Edit it and re-run: the buckets, counts and verdicts below are derived from it, not hard-coded.

**Output** — the opening lines of a real run:

```
    when  event                       drift  new tools
------------------------------------------------------------------
    90d  control signed off          0.000  []
    60d  prompt edited               0.100  []
    30d  tool added (no PR)          0.330  ['run_shell']
     5d  model upgraded by vendor    0.550  ['run_shell']
observed drift rate  0.00647 TVD/day
tolerance            0.25
```

The run continues past this. The script is the example: `test_skills.py` executes it on every build, so this block cannot drift from what the skill actually prints.

## Output contract

```json
{
  "surfaces": [{"name": "str", "produces_change_record": false}],
  "bypass_count": 0,
  "baseline": {"at": "str", "tools": ["str"], "model_version": "str", "prompt_hash": "str"},
  "timeline": [{"period": "str", "drift": 0.0, "new": ["str"], "attributed_to": "str|null"}],
  "tolerance": 0.0,
  "consequence": "str"
}
```

## Failure modes

- **Trusting the changelog.** Most of what changes an agent is not in it.
- **Drift with no attribution.** A number nobody can act on.
- **No consequence at the tolerance.** The line stops being a line.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/detection/behavioural-drift-monitor/scripts/behavioural_drift_monitor.py
SCRIPT = "skills/detection/behavioural-drift-monitor/scripts/behavioural_drift_monitor.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

Drift rises across the quarter from 0.0 at sign-off to roughly 0.35 after the model upgrade, with `run_shell` appearing as a new tool. Four of six change surfaces bypass change management. The observed drift rate yields a freshness window, and the 90-day-old control test is reported STALE rather than passing.

## Your turn

Compute the drift rate for one production agent from three months of telemetry, and set its control freshness window from that number rather than from the audit calendar.

---

**Next → [D1.8 · Threat intel sub-lane](https://spbreed.github.io/cyber-commons/lessons/D1.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*